# Seoul English Education Expansion Opportunity Index

This project develops a decision-support tool for identifying potential expansion areas for NADO's in-person English education service across 425 administrative neighborhoods in Seoul.

The index combines 1)local adult learning demand, 2)the availability and price of existing English education, and 3)public-transit accessibility.

A higher score indicates a stronger expansion opportunity: relatively high potential demand, limited existing educational supply, higher competing tuition prices, and stronger operational accessibility by public transit.

## Research Question

Which administrative neighborhoods in Seoul offer the strongest expansion opportunities for NADO, considering potential adult demand, existing English education supply, listed tuition costs, and public-transit accessibility?

The index consists of three pillars:

1. **Demand–Supply Opportunity**  
   Measures potential adult learning demand relative to the number of existing English education institutions. Fewer institutions relative to the target population indicate a larger potential market gap.

2. **Price Opportunity**  
   Measures the typical publicly listed tuition level of existing English education institutions. Higher local tuition may provide NADO with greater potential for price differentiation.

3. **Operational Accessibility**  
   Measures bus and subway accessibility within each administrative neighborhood. Better transit access may make it easier for NADO teachers and students to participate in in-person lessons.

The primary model uses the living population aged 40–69, reflecting the age profile most relevant to NADO's current offline service. A broader 20–69 model is used as a sensitivity analysis.

## 1. Libraries

In [ ]:
# Data processing
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

# Visualization
import matplotlib.pyplot as plt

## 2. Data Sources and Loading

### 2.1 Source Data

Four public datasets are used:

- **Seoul Living Population:** hourly dong population by age and sex, used to measure potential demand.
- **NEIS Academy and Tutoring Center Data:** institution type, course information, address, and publicly listed tuition, used to measure supply and cost.
- **Seoul Bus Stop Location Data:** longitude and latitude of bus stops, used to measure local transit accessibility.
- **Seoul Station Master Data:** longitude and latitude of subway and rail stations, used to measure metropolitan transit accessibility.

The dong boundary SHP file is used as the master geographic unit for all spatial joins.

#### 2.1.1 Seoul Living Population

In [ ]:
ppl = pd.read_csv('./[내국인] 행정동별 서울 생활인구(250m).csv', encoding='cp949',low_memory=False)
#생활인구(250m) : https://data.seoul.go.kr/dataList/OA-23016/S/1/datasetView.do
#for potential demand

#### 2.1.2 NEIS Academy and Tutoring Center Data

In [ ]:
admy2 = pd.read_csv('./학원교습소정보_2026년8월31일기준.csv', encoding='cp949', low_memory=False)
#neis 원본 자료 https://open.neis.go.kr/portal/data/service/selectServicePage.do?page=1&rows=10&sortColumn=&sortDirection=&infId=OPEN19220231012134453534385&infSeq=3
#korea's nationwide acadamy 원자료 >> select seoul data
#in address form
#for supply & payment willingness

#### 2.1.3 Seoul Bus Stop Location Data

In [ ]:
bus = pd.read_excel('./서울시버스정류소위치정보(20260902).xlsx')
#버스 : https://data.seoul.go.kr/dataList/OA-15067/S/1/datasetView.do
#seoul bus station point and type
#in x,y point
#for physical accessibility

#### 2.1.4 Seoul Station Master Data

In [ ]:
metro = pd.read_csv('./서울시 역사마스터 정보.csv', encoding='cp949')
#지하철 : https://data.seoul.go.kr/dataList/OA-21232/A/1/datasetView.do
#capital-near metro station point >> select seoul data

### 2.2 Initial Data Audit

In [ ]:
datasets = {
    "Living Population": ppl,
    "Academies": admy2,
    "Bus Stops": bus,
    "Metro Stations": metro
}

data_summary = pd.DataFrame(
    [
        {
            "Dataset": name,
            "Rows": len(df),
            "Columns": len(df.columns),
            "Missing Cells": int(df.isna().sum().sum())
        }
        for name, df in datasets.items()
    ]
)

display(data_summary)

## 3. Academy Data Preparation

The nationwide NEIS dataset is filtered to institutions that:

- are registered with the Seoul Metropolitan Office of Education
- are currently open
- have a valid road-name address

Text columns are stripped of unnecessary whitespace.  
The tuition column is renamed to '인당수강료내용', and the road-name address is retained as '검색주소' for geocoding.

In [ ]:
# 기준 학원 데이터 생성: admy2 사용
academy = (
    
    admy2.loc[
        admy2["시도교육청명"].eq("서울특별시교육청")  #select acadamies in seoul
        & admy2["등록상태명"].eq("개원")          #currently open
        & admy2["도로명주소"].notna()            #have a valid road-name address
    ] 
    .rename(columns={                         #change name for easier recognition
        "학원교습소명": "학원/교습소",
        "인당수강료": "인당수강료내용"
    })
    .copy()
)

for col in academy.columns:
    if (
        pd.api.types.is_string_dtype(academy[col].dtype)
        or pd.api.types.is_object_dtype(academy[col].dtype)
    ):
        academy[col] = academy[col].astype("string").str.strip() #column type : ? -> string & delete white space

academy["인당수강료내용"] = (
    academy["인당수강료내용"].replace("", pd.NA)    # " " -> NA
)

academy["검색주소"] = (
    academy["도로명주소"].astype("string").str.strip()
)

## 4. Identification of English Education Institutions

The NEIS dataset does not provide a single variable identifying English education institutions. Therefore, institution names, curriculum fields, course names, and listed tuition descriptions are combined into one searchable text field.

Keyword-based rules are then used to identify:

- confirmed English education institutions
- institutions that may offer adult English conversation
- youth-focused institutions
- non-English language institutions
- remote institutions

The main supply indicator uses all confirmed English education institutions. Adult-related classifications are retained as supplementary evidence because adult-specific course information is limited.

In [ ]:
# 학원 분류에 사용할 텍스트 열
text_columns = [
    "학원명",
    "분야명",
    "교습계열명",
    "교습과정목록명",
    "교습과정명",
    "인당수강료내용"
]

# 실제 데이터에 존재하는 열만 사용
text_columns = [
    col
    for col in text_columns
    if col in academy.columns
]

# 여러 텍스트 열을 하나의 검색용 문자열로 결합 (keyword-based rules 접목 한번에 하기 위해)
academy["분류텍스트_확장"] = (
    academy[text_columns]
    .fillna("")
    .astype("string")
    .agg(" ".join, axis=1)       #aggregate all columns in one column
    .str.lower()
)

print("분류에 사용한 열:", text_columns)

In [ ]:
text = academy["분류텍스트_확장"]

# 영어 관련 표현                 #check if each row has search keywords with results in boolean
academy["영어표시_확장"] = text.str.contains( 
    r"영어|english|잉글리시|"          #r means string. 
    r"토익|toeic|토플|toefl|"         #| means or
    r"오픽|opic|아이엘츠|ielts",
    case=False,
    na=False
)

# 성인 영어회화 가능성이 높은 표현
academy["성인영어회화_강한표시"] = text.str.contains(
    r"성인영어|성인회화|영어회화|"
    r"비즈니스영어|비즈니스회화|"
    r"business english|english conversation|"
    r"프리토킹",
    case=False,
    na=False
)

# 성인 어학 과정
academy["성인어학과정표시"] = text.str.contains(
    r"어학\(성인\)",
    case=False,
    na=False
)

# 아동·청소년 대상 표현
academy["아동청소년표시"] = text.str.contains(
    r"유아|초등|중등|고등|"
    r"초[·ㆍ]?중[·ㆍ]?고|"
    r"파닉스|입시|보습",
    case=False,
    na=False
)

# 영어 이외의 외국어 표현
academy["비영어언어표시"] = text.str.contains(
    r"중국어|일본어|프랑스어|독일어|"
    r"스페인어|러시아어|이탈리아어|"
    r"베트남어|한국어|코리안|"
    r"eju|hsk|jlpt|delf|topik",
    case=False,
    na=False
)

# 원격수업 기관
academy["원격표시"] = text.str.contains(
    r"원격",
    case=False,
    na=False
)

In [ ]:
# 다른 언어는 있지만 영어 표시는 없는 기관    # search not eng related kw
academy["비영어전용표시"] = (
    academy["비영어언어표시"]
    & ~academy["영어표시_확장"]       # ! = NOT : True & False reverses.
)

# 아동·청소년 표현은 있지만 성인 과정 표시는 없는 기관
academy["아동청소년전용표시"] = (      # exclude adult edu
    academy["아동청소년표시"]
    & ~academy["성인어학과정표시"]
)

# 최종 영어교육기관
academy["영어기관_자동"] = (         # exclude online edu
    academy["영어표시_확장"]
    & ~academy["원격표시"]
)

# 성인 영어회화 가능성이 높은 기관
academy["성인영어회화_엄격"] = (    
    (
        academy["성인영어회화_강한표시"]
        | (
            academy["성인어학과정표시"]
            & academy["영어표시_확장"]
        )
    )
    & ~academy["원격표시"]
    & ~academy["비영어전용표시"]
    & ~academy["아동청소년전용표시"]
)

# 성인 어학기관이지만 언어가 명확하지 않은 기관
academy["성인어학_정보부족"] = (
    academy["성인어학과정표시"]
    & ~academy["영어표시_확장"]
    & ~academy["비영어전용표시"]
    & ~academy["원격표시"]
)

In [ ]:
classification_summary = pd.DataFrame(
    {
        "Classification": [
            "Confirmed English institutions",
            "Strict adult English conversation",
            "Adult language with insufficient language information",
            "Remote institutions"
        ],
        "Number of institutions": [
            int(academy["영어기관_자동"].sum()),
            int(academy["성인영어회화_엄격"].sum()),
            int(academy["성인어학_정보부족"].sum()),
            int(academy["원격표시"].sum())
        ]
    }
)

display(classification_summary)

### 4.1 Addresses Selected for Geocoding

Confirmed English institutions and supplementary adult-language candidates are retained for the one-time geocoding step. The final supply indicator counts only confirmed English institutions.

In [ ]:
academy_geo_target = academy.loc[
    academy["영어기관_자동"]
    | academy["성인영어회화_엄격"]
    | academy["성인어학_정보부족"]
].copy()

academy_geo_target = academy_geo_target.drop_duplicates(
    subset="학원지정번호"
)

academy_geo_target = academy_geo_target.loc[
    academy_geo_target["검색주소"].notna()
    & academy_geo_target["검색주소"].ne("")
].copy()

address_lookup = (
    academy_geo_target[["검색주소"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("좌표 변환 대상 학원 수:", len(academy_geo_target))
print("고유 주소 수:", len(address_lookup))
print(
    "중복 주소로 절약되는 API 요청:",
    len(academy_geo_target) - len(address_lookup)
)
display(address_lookup.head(10))

## 5. Geocoding Academy Addresses

Academy road-name addresses were converted into longitude and latitude coordinates using the SGIS geocoding API.

Sections 5.1–5.5 document the one-time geocoding workflow. These cells are retained as Raw cells to prevent repeated API calls during `Run All`.

The main analysis resumes in Section 5.6 by loading the saved `sgis_geocode_checkpoint.csv` file.

### 5.1 SGIS API Authentication

This step requests a temporary SGIS access token using the service ID and security key. `getpass()` prevents the credentials from being displayed or stored directly in the notebook.

### 5.2 Single-Address Validation

Before processing all addresses, one road-name address is sent to SGIS to verify that 1)authentication, 2)address matching, and 3)WGS84 coordinate retrieval work correctly.

### 5.3 Checkpoint Setup and Pending Addresses

The geocoding process may stop because of API limits, token expiration, or network errors. This step loads any previously saved results and identifies only the addresses that have not yet been processed.

### 5.4 Geocoding Function

This function sends one address to SGIS and returns its longitude, latitude, administrative information, matching status, and processing status. It also handles token expiration, failed matches, and temporary request errors.

### 5.5 Batch Geocoding and Checkpoint Saving

The remaining addresses are geocoded sequentially. Results are saved every 100 addresses so that completed work is preserved even if the process is interrupted.

In [ ]:
checkpoint_path = Path("sgis_geocode_checkpoint.csv")

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"지오코딩 결과 파일이 없습니다: {checkpoint_path.resolve()}"
    )

geocode_results = pd.read_csv(
    checkpoint_path,
    dtype={"행정동코드": "string"}
)

geocode_results["검색주소"] = (
    geocode_results["검색주소"]
    .astype("string")
    .str.strip()
)

geocode_results = geocode_results.drop_duplicates(
    subset="검색주소",
    keep="last"
)

missing_addresses = address_lookup.loc[
    ~address_lookup["검색주소"].isin(geocode_results["검색주소"])
]

assert len(missing_addresses) == 0
print("지오코딩 결과 불러오기 완료:", len(geocode_results))

### 5.6 Load Saved Coordinates and Merge with Academy Data

The saved geocoding results are loaded from `sgis_geocode_checkpoint.csv`. Longitude and latitude are converted to numeric values and joined to the academy dataset using the road-name address.

SGIS administrative-dong codes are retained only for validation. Final neighborhood assignment is performed through a spatial join with the project SHP boundary.

In [ ]:
geocode_results["경도"] = pd.to_numeric(
    geocode_results["경도"], errors="coerce"
)
geocode_results["위도"] = pd.to_numeric(
    geocode_results["위도"], errors="coerce"
)
geocode_results["행정동코드"] = (
    geocode_results["행정동코드"].astype("string").str.strip()
)

academy_geo = academy_geo_target.merge(
    geocode_results,
    on="검색주소",
    how="left",
    validate="many_to_one"
)

assert len(academy_geo) == len(academy_geo_target)
assert academy_geo["경도"].notna().all()
assert academy_geo["위도"].notna().all()

print("대상 학원 수:", len(academy_geo))
print("좌표 확보 학원 수:", academy_geo["경도"].notna().sum())
display(
    academy_geo[
        ["학원명", "검색주소", "경도", "위도", "행정동명", "지오코딩상태"]
    ].head()
)

## 6. Spatial Assignment to Administrative Dongs

The administrative-dong SHP file defines the 425 geographic units used throughout the analysis.

Academy longitude and latitude coordinates are converted into geographic point objects. Each point is then assigned to the SHP polygon that contains it.

This spatial join is used instead of directly merging SGIS administrative codes because SGIS and the SHP file use different administrative-code systems.

In [ ]:
# SHP 파일 탐색 및 불러오기
shp_files = glob.glob(
    "./서울시 상권분석서비스(영역-행정동)/**/*.shp",
    recursive=True
)

if not shp_files:
    raise FileNotFoundError(
        "행정동 SHP 파일을 찾을 수 없습니다."
    )

dong_map = gpd.read_file(shp_files[0])

dong_map["ADSTRD_CD"] = (
    dong_map["ADSTRD_CD"]
    .astype("string")
    .str.strip()
)

print("SHP 파일:", shp_files[0])
print("SHP 좌표계:", dong_map.crs)
print("SHP 행정동 수:", len(dong_map))

In [ ]:
academy_gdf = gpd.GeoDataFrame(
    academy_geo.copy(),
    geometry=gpd.points_from_xy(
        academy_geo["경도"],
        academy_geo["위도"]
    ),
    crs="EPSG:4326"
)

In [ ]:
academy_gdf = academy_gdf.to_crs(
    dong_map.crs
)

print("학원 좌표계:", academy_gdf.crs)
print("SHP 좌표계:", dong_map.crs)

### 6.2 Spatial Join

A spatial join assigns each academy point to the administrative-dong polygon that contains it. The SHP code `ADSTRD_CD` becomes the final geographic key used throughout the index.

In [ ]:
dong_boundary = dong_map[
    [
        "ADSTRD_CD",
        "ADSTRD_NM",
        "geometry"
    ]
].copy()

academy_by_shp = gpd.sjoin(
    academy_gdf,
    dong_boundary,
    how="left",
    predicate="within"
)

In [ ]:
spatial_join_summary = pd.Series(
    {
        "Original academies": len(academy_gdf),
        "Rows after spatial join": len(academy_by_shp),
        "Successfully matched": int(academy_by_shp["ADSTRD_CD"].notna().sum()),
        "Unmatched": int(academy_by_shp["ADSTRD_CD"].isna().sum()),
        "Duplicate academy IDs": int(
            academy_by_shp.duplicated(subset="학원지정번호").sum()
        )
    }
)

display(spatial_join_summary.to_frame(name="Count"))

assert len(academy_by_shp) == len(academy_gdf)
assert academy_by_shp["ADSTRD_CD"].notna().all()
assert not academy_by_shp.duplicated(subset="학원지정번호").any()

academy_by_shp = academy_by_shp.drop(
    columns="index_right",
    errors="ignore"
)

In [ ]:
display(
    academy_by_shp[
        [
            "학원명",
            "검색주소",
            "행정동명",
            "ADSTRD_CD",
            "ADSTRD_NM"
        ]
    ].sample(
        10,
        random_state=42
    )
)

### 6.3 Aggregate Education Supply by Administrative Dong

Academies are grouped by their SHP administrative-dong code. Boolean classification variables are summed to calculate the number of English education institutions in each neighborhood.

`영어기관수` is the primary supply measure. Adult-specific counts are retained only as supplementary diagnostics.

In [ ]:
academy_by_shp["ADSTRD_CD"] = (
    academy_by_shp["ADSTRD_CD"]
    .astype("string")
    .str.strip()
)

dong_supply = (
    academy_by_shp
    .groupby(
        ["ADSTRD_CD", "ADSTRD_NM"],
        as_index=False
    )
    .agg(
        영어기관수=(
            "영어기관_자동",
            "sum"
        ),
        성인영어회화_엄격수=(
            "성인영어회화_엄격",
            "sum"
        ),
        성인어학_정보부족수=(
            "성인어학_정보부족",
            "sum"
        )
    )
)

dong_supply["성인어학_넓은후보수"] = (
    dong_supply["성인영어회화_엄격수"]
    + dong_supply["성인어학_정보부족수"]
)

count_columns = [
    "영어기관수",
    "성인영어회화_엄격수",
    "성인어학_정보부족수",
    "성인어학_넓은후보수"
]

dong_supply[count_columns] = (
    dong_supply[count_columns].astype(int)
)

display(
    dong_supply.sort_values(
        "영어기관수",
        ascending=False
    ).head(30)
)

In [ ]:
assert (
    dong_supply["영어기관수"].sum()
    == academy_by_shp["영어기관_자동"].sum()
)

assert (
    dong_supply["성인영어회화_엄격수"].sum()
    == academy_by_shp["성인영어회화_엄격"].sum()
)

print("행정동별 공급 집계 검증 통과")

### 6.4 Create the Master Geographic Dataset

The SHP GeoDataFrame is copied to create `index_base`, the master dataset containing one row for each of Seoul's 425 administrative neighborhoods.

Supply, demand, cost, and transit variables are subsequently merged into this dataset using `ADSTRD_CD`.

In [ ]:
index_base = dong_map.copy()

index_base["ADSTRD_CD"] = (
    index_base["ADSTRD_CD"]
    .astype("string")
    .str.strip()
)

index_base = index_base.merge(
    dong_supply.drop(columns="ADSTRD_NM"),
    on="ADSTRD_CD",
    how="left",
    validate="one_to_one"
)

index_base[count_columns] = (
    index_base[count_columns]
    .fillna(0)
    .astype(int)
)

print("SHP 행정동:", len(dong_map))
print("index_base 행정동:", len(index_base))
print("영어기관 합계:", index_base["영어기관수"].sum())

## 7. Adult Living-Population Demand

Seoul living-population data estimates the number of people present in each administrative dong by date, hour, age, and sex.

Two adult-demand populations are calculated:

- **40–69 population:** the primary target population for the NADO expansion model, reflecting the age profile most relevant to the current offline service;
- **20–69 population:** a broader adult population used for sensitivity analysis.

The mean hourly population over the observation period is used instead of the sum. Summing hourly observations would count the same population repeatedly across dates and hours.

In [ ]:
population_dates = pd.to_datetime(
    ppl["일자"].astype("string"),
    format="%Y%m%d"
)

population_period_summary = pd.Series(
    {
        "Start date": population_dates.min(),
        "End date": population_dates.max(),
        "Number of dates": population_dates.nunique(),
        "Number of hours": ppl["시간"].nunique(),
        "Number of administrative dongs": (
            ppl["행정동코드"].nunique()
        ),
        "Total rows": len(ppl)
    }
)

display(
    population_period_summary.to_frame(
        name="Value"
    )
)

In [ ]:
age_20_69_columns = [
    "남자 20~24세", "남자 25~29세",
    "남자 30~34세", "남자 35~39세",
    "남자 40~44세", "남자 45~49세",
    "남자 50~54세", "남자 55~59세",
    "남자 60~64세", "남자 65~69세",
    "여자 20~24세", "여자 25~29세",
    "여자 30~34세", "여자 35~39세",
    "여자 40~44세", "여자 45~49세",
    "여자 50~54세", "여자 55~59세",
    "여자 60~64세", "여자 65~69세"
]

age_40_69_columns = [
    "남자 40~44세", "남자 45~49세",
    "남자 50~54세", "남자 55~59세",
    "남자 60~64세", "남자 65~69세",
    "여자 40~44세", "여자 45~49세",
    "여자 50~54세", "여자 55~59세",
    "여자 60~64세", "여자 65~69세"
]

target_age_columns = list(
    dict.fromkeys(age_20_69_columns + age_40_69_columns)
)

missing_age_columns = [
    col for col in target_age_columns
    if col not in ppl.columns
]

if missing_age_columns:
    raise KeyError(
        f"생활인구 자료에 없는 연령 열: {missing_age_columns}"
    )

ppl[target_age_columns] = ppl[target_age_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

ppl["행정동코드"] = (
    ppl["행정동코드"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

ppl["생활인구_20_69"] = ppl[
    age_20_69_columns
].sum(axis=1, min_count=1)

ppl["생활인구_40_69"] = ppl[
    age_40_69_columns
].sum(axis=1, min_count=1)

In [ ]:
demand_by_dong = (
    ppl.groupby("행정동코드", as_index=False)
    .agg(
        생활인구_20_69=("생활인구_20_69", "mean"),
        생활인구_40_69=("생활인구_40_69", "mean")
    )
)

print("생활인구 기준 행정동 수:", len(demand_by_dong))
display(demand_by_dong.head())

In [ ]:
code_to_map = {
    "11230515": "11230536",
    "11230533": "11230536",
    "11680675": "11680740",
    "11740525": "11740520",
    "11740526": "11740515"
}

demand_by_dong["ADSTRD_CD"] = (
    demand_by_dong["행정동코드"]
    .replace(code_to_map)
)

demand_shp = (
    demand_by_dong
    .groupby("ADSTRD_CD", as_index=False)
    .agg(
        생활인구_20_69=("생활인구_20_69", "sum"),
        생활인구_40_69=("생활인구_40_69", "sum")
    )
)

shp_codes = set(dong_map["ADSTRD_CD"])
demand_codes = set(demand_shp["ADSTRD_CD"])

print("SHP에만 있는 코드:", shp_codes - demand_codes)
print("생활인구에만 있는 코드:", demand_codes - shp_codes)
print("SHP 행정동 수:", len(dong_map))
print("정리된 생활인구 행정동 수:", len(demand_shp))

assert shp_codes == demand_codes

In [ ]:
index_base = index_base.drop(
    columns=["생활인구_20_69", "생활인구_40_69"],
    errors="ignore"
)

index_base = index_base.merge(
    demand_shp[
        ["ADSTRD_CD", "생활인구_20_69", "생활인구_40_69"]
    ],
    on="ADSTRD_CD",
    how="left",
    validate="one_to_one"
)

assert len(index_base) == 425
assert index_base["생활인구_20_69"].notna().all()
assert index_base["생활인구_40_69"].notna().all()

print("생활인구 결합 후 행정동 수:", len(index_base))

In [ ]:
index_base["영어기관_1만명당_20_69"] = (
    index_base["영어기관수"]
    / index_base["생활인구_20_69"]
    * 10_000
)

index_base["영어기관_1만명당_40_69"] = (
    index_base["영어기관수"]
    / index_base["생활인구_40_69"]
    * 10_000
)

display(
    index_base[
        [
            "ADSTRD_NM", "영어기관수", "생활인구_20_69",
            "생활인구_40_69", "영어기관_1만명당_20_69",
            "영어기관_1만명당_40_69"
        ]
    ].head(10)
)

## 8. Pillar 2: Price Opportunity

Public tuition strings are parsed into course-level prices. A median is first calculated for each institution and then for each administrative dong, preventing institutions with many listed courses from receiving disproportionate weight.

Zero- and one-won placeholders are excluded. Missing dong-level tuition values are imputed with the corresponding district median while retaining an imputation flag.

In [ ]:
fee_source_columns = [
    "학원지정번호",
    "학원명",
    "ADSTRD_CD",
    "ADSTRD_NM",
    "교습과정명",
    "인당수강료내용",
    "영어기관_자동",
    "성인영어회화_엄격",
    "성인어학_정보부족"
]

fee_source = academy_by_shp.loc[
    academy_by_shp["영어기관_자동"],
    fee_source_columns
].copy()

fee_source["인당수강료내용"] = (
    fee_source["인당수강료내용"]
    .fillna("")
    .astype("string")
    .str.strip()
)

fee_source = fee_source.loc[
    fee_source["인당수강료내용"].ne("")
].copy()

print("전체 영어교육기관:", len(academy_by_shp))
print("수강료 정보가 있는 기관:", len(fee_source))
print(
    "수강료 공개 비율:",
    round(len(fee_source) / len(academy_by_shp) * 100, 1),
    "%"
)

display(
    fee_source[
        [
            "학원명",
            "ADSTRD_NM",
            "교습과정명",
            "인당수강료내용"
        ]
    ].head(10)
)

In [ ]:
fee_pattern = (
    r"(?:^|,\s*)"
    r"(?P<수강과정명>.*?)"
    r":\s*"
    r"(?P<수강료>[0-9][0-9,]*)"
    r"\s*(?:원)?"
    r"(?=,|$)"
)

fee_extracted = (
    fee_source
    .set_index("학원지정번호")["인당수강료내용"]
    .str.extractall(fee_pattern)
    .reset_index()
)

fee_extracted["수강과정명"] = (
    fee_extracted["수강과정명"]
    .astype("string")
    .str.strip()
)

fee_extracted["수강료"] = pd.to_numeric(
    fee_extracted["수강료"]
    .str.replace(",", "", regex=False),
    errors="coerce"
)

fee_course = fee_extracted.merge(
    fee_source.drop(columns="인당수강료내용"),
    on="학원지정번호",
    how="left",
    validate="many_to_one"
)

print("수강료 공개 기관:", len(fee_source))
print("분리된 과정·수강료 행:", len(fee_course))
print("금액 변환 실패:", fee_course["수강료"].isna().sum())

display(
    fee_course[
        [
            "학원명",
            "ADSTRD_NM",
            "수강과정명",
            "수강료"
        ]
    ].head(20)
)

In [ ]:
parsed_academy_ids = set(
    fee_course["학원지정번호"].dropna()
)

unparsed_fee_source = fee_source.loc[
    ~fee_source["학원지정번호"].isin(parsed_academy_ids)
].copy()

print("수강료 문자열 파싱 실패 기관:", len(unparsed_fee_source))

display(
    unparsed_fee_source[
        [
            "학원명",
            "인당수강료내용"
        ]
    ].head(20)
)

In [ ]:
print(
    fee_course["수강료"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
)

print(
    "0원 과정:",
    fee_course["수강료"].eq(0).sum()
)

print(
    "1만 원 미만:",
    fee_course["수강료"].lt(10_000).sum()
)

print(
    "200만 원 초과:",
    fee_course["수강료"].gt(2_000_000).sum()
)

display(
    fee_course.loc[
        (fee_course["수강료"] <= 0)
        | (fee_course["수강료"] > 2_000_000),
        [
            "학원명",
            "ADSTRD_NM",
            "수강과정명",
            "수강료"
        ]
    ].sort_values("수강료")
)

In [ ]:
fee_course["수강료_품질"] = np.select(
    [
        fee_course["수강료"].eq(0),
        fee_course["수강료"].eq(1),
        fee_course["수강료"].between(2, 9_999),
        fee_course["수강료"].gt(2_000_000)
    ],
    [
        "0원",
        "1원_placeholder",
        "1만원미만_확인필요",
        "200만원초과_고가"
    ],
    default="일반범위"
)

print(fee_course["수강료_품질"].value_counts())

In [ ]:
fee_valid = fee_course.loc[
    fee_course["영어기관_자동"]
    & fee_course["수강료"].notna()
    & fee_course["수강료"].gt(1)
].copy()

print("전체 과정 가격:", len(fee_course))
print("분석에 사용할 영어 과정 가격:", len(fee_valid))
print("제외된 0원·1원 가격:", len(fee_course) - len(fee_valid))

In [ ]:
academy_fee = (
    fee_valid
    .groupby(
        [
            "학원지정번호",
            "학원명",
            "ADSTRD_CD",
            "ADSTRD_NM"
        ],
        as_index=False
    )
    .agg(
        학원대표수강료=("수강료", "median"),
        공개과정수=("수강료", "size"),
        학원최저수강료=("수강료", "min"),
        학원최고수강료=("수강료", "max")
    )
)

print("대표 수강료가 계산된 학원:", len(academy_fee))

display(
    academy_fee.sort_values(
        "학원대표수강료",
        ascending=False
    ).head(20)
)

In [ ]:
dong_fee = (
    academy_fee
    .groupby(
        ["ADSTRD_CD", "ADSTRD_NM"],
        as_index=False
    )
    .agg(
        행정동_중위수강료=("학원대표수강료", "median"),
        수강료공개기관수=("학원지정번호", "nunique"),
        수강료_Q1=(
            "학원대표수강료",
            lambda x: x.quantile(0.25)
        ),
        수강료_Q3=(
            "학원대표수강료",
            lambda x: x.quantile(0.75)
        )
    )
)

print("수강료가 확인된 행정동:", len(dong_fee))

display(
    dong_fee.sort_values(
        "행정동_중위수강료",
        ascending=False
    ).head(20)
)

In [ ]:
fee_columns = [
    "행정동_중위수강료",
    "수강료공개기관수",
    "수강료_Q1",
    "수강료_Q3"
]

# 같은 셀을 재실행할 때 중복 열 방지
index_base = index_base.drop(
    columns=fee_columns + ["수강료공개율"],
    errors="ignore"
)

index_base = index_base.merge(
    dong_fee.drop(columns="ADSTRD_NM"),
    on="ADSTRD_CD",
    how="left",
    validate="one_to_one"
)

index_base["수강료공개기관수"] = (
    index_base["수강료공개기관수"]
    .fillna(0)
    .astype(int)
)

index_base["수강료공개율"] = np.where(
    index_base["영어기관수"].gt(0),
    index_base["수강료공개기관수"]
    / index_base["영어기관수"],
    np.nan
)

print(
    "수강료가 없는 행정동:",
    index_base["행정동_중위수강료"].isna().sum()
)

print(
    index_base["행정동_중위수강료"].describe()
)

display(
    index_base[
        [
            "ADSTRD_NM",
            "영어기관수",
            "수강료공개기관수",
            "수강료공개율",
            "행정동_중위수강료"
        ]
    ].sort_values(
        "수강료공개율"
    ).head(20)
)

In [ ]:
gu_name_map = {
    "11110": "종로구",
    "11140": "중구",
    "11170": "용산구",
    "11200": "성동구",
    "11215": "광진구",
    "11230": "동대문구",
    "11260": "중랑구",
    "11290": "성북구",
    "11305": "강북구",
    "11320": "도봉구",
    "11350": "노원구",
    "11380": "은평구",
    "11410": "서대문구",
    "11440": "마포구",
    "11470": "양천구",
    "11500": "강서구",
    "11530": "구로구",
    "11545": "금천구",
    "11560": "영등포구",
    "11590": "동작구",
    "11620": "관악구",
    "11650": "서초구",
    "11680": "강남구",
    "11710": "송파구",
    "11740": "강동구"
}

index_base["구코드"] = (
    index_base["ADSTRD_CD"]
    .astype("string")
    .str[:5]
)

index_base["자치구명"] = (
    index_base["구코드"]
    .map(gu_name_map)
)

academy_fee["구코드"] = (
    academy_fee["ADSTRD_CD"]
    .astype("string")
    .str[:5]
)

assert index_base["자치구명"].notna().all()

print("자치구 수:", index_base["자치구명"].nunique())

In [ ]:
gu_fee = (
    academy_fee
    .groupby("구코드", as_index=False)
    .agg(
        자치구_중위수강료=(
            "학원대표수강료",
            "median"
        ),
        자치구_수강료공개기관수=(
            "학원지정번호",
            "nunique"
        )
    )
)

gu_fee["자치구명"] = gu_fee["구코드"].map(
    gu_name_map
)

display(
    gu_fee.sort_values(
        "자치구_중위수강료",
        ascending=False
    )
)

In [ ]:
index_base = index_base.drop(
    columns=[
        "자치구_중위수강료",
        "자치구_수강료공개기관수",
        "수강료_분석값",
        "수강료_대체수준"
    ],
    errors="ignore"
)

index_base = index_base.merge(
    gu_fee[
        [
            "구코드",
            "자치구_중위수강료",
            "자치구_수강료공개기관수"
        ]
    ],
    on="구코드",
    how="left",
    validate="many_to_one"
)

print(
    "자치구 중위 수강료 결측:",
    index_base["자치구_중위수강료"].isna().sum()
)

In [ ]:
seoul_median_fee = academy_fee[
    "학원대표수강료"
].median()

index_base["수강료_분석값"] = (
    index_base["행정동_중위수강료"]
    .fillna(index_base["자치구_중위수강료"])
    .fillna(seoul_median_fee)
)

index_base["수강료_대체수준"] = np.select(
    [
        index_base["행정동_중위수강료"].notna(),
        (
            index_base["행정동_중위수강료"].isna()
            & index_base["자치구_중위수강료"].notna()
        )
    ],
    [
        "행정동 관측값",
        "자치구 중위값 대체"
    ],
    default="서울 중위값 대체"
)

print(
    index_base["수강료_대체수준"].value_counts()
)

print(
    "최종 수강료 결측:",
    index_base["수강료_분석값"].isna().sum()
)

In [ ]:
index_base["수강료_자료신뢰도"] = np.select(
    [
        index_base["행정동_중위수강료"].isna(),
        index_base["수강료공개기관수"].eq(1),
        index_base["수강료공개기관수"].between(2, 4),
        index_base["수강료공개기관수"].ge(5)
    ],
    [
        "대체값",
        "낮음",
        "보통",
        "높음"
    ],
    default="확인필요"
)

print(
    index_base["수강료_자료신뢰도"].value_counts()
)

display(
    index_base[
        [
            "자치구명",
            "ADSTRD_NM",
            "영어기관수",
            "수강료공개기관수",
            "행정동_중위수강료",
            "자치구_중위수강료",
            "수강료_분석값",
            "수강료_대체수준",
            "수강료_자료신뢰도"
        ]
    ].head(20)
)

## 9. Pillar 3: Operational Accessibility

Bus stops and station-line records are spatially joined to the same 425 SHP units. Counts are divided by polygon area to measure transit density. Higher density represents stronger operational accessibility for in-person service delivery.

In [ ]:
bus_clean = bus.copy()

bus_clean["X좌표"] = pd.to_numeric(
    bus_clean["X좌표"],
    errors="coerce"
)

bus_clean["Y좌표"] = pd.to_numeric(
    bus_clean["Y좌표"],
    errors="coerce"
)

bus_clean = bus_clean.loc[
    bus_clean["X좌표"].notna()
    & bus_clean["Y좌표"].notna()
    & bus_clean["X좌표"].between(126, 128)
    & bus_clean["Y좌표"].between(37, 38)
    & bus_clean["정류소타입"].ne("한강선착장")
].copy()

bus_clean = bus_clean.drop_duplicates(
    subset="NODE_ID"
)

bus_gdf = gpd.GeoDataFrame(
    bus_clean,
    geometry=gpd.points_from_xy(
        bus_clean["X좌표"],
        bus_clean["Y좌표"]
    ),
    crs="EPSG:4326"
)

bus_gdf = bus_gdf.to_crs(dong_map.crs)

print("분석 대상 버스정류소:", len(bus_gdf))
print("버스 좌표계:", bus_gdf.crs)

In [ ]:
bus_by_shp = gpd.sjoin(
    bus_gdf,
    dong_boundary,
    how="left",
    predicate="within"
)

print("전체 버스정류소:", len(bus_by_shp))
print(
    "행정동 연결 성공:",
    bus_by_shp["ADSTRD_CD"].notna().sum()
)
print(
    "행정동 연결 실패:",
    bus_by_shp["ADSTRD_CD"].isna().sum()
)
print(
    "중복 NODE_ID:",
    bus_by_shp.duplicated("NODE_ID").sum()
)

display(
    bus_by_shp.loc[
        bus_by_shp["ADSTRD_CD"].isna(),
        [
            "NODE_ID",
            "정류소명",
            "X좌표",
            "Y좌표",
            "정류소타입"
        ]
    ]
)

In [ ]:
metro_clean = metro.copy()

metro_clean["경도"] = pd.to_numeric(
    metro_clean["경도"],
    errors="coerce"
)

metro_clean["위도"] = pd.to_numeric(
    metro_clean["위도"],
    errors="coerce"
)

metro_clean = metro_clean.loc[
    metro_clean["경도"].notna()
    & metro_clean["위도"].notna()
    & metro_clean["경도"].between(126, 128)
    & metro_clean["위도"].between(37, 38.5)
].copy()

metro_clean = metro_clean.drop_duplicates(
    subset="역사_ID"
)

metro_gdf = gpd.GeoDataFrame(
    metro_clean,
    geometry=gpd.points_from_xy(
        metro_clean["경도"],
        metro_clean["위도"]
    ),
    crs="EPSG:4326"
)

metro_gdf = metro_gdf.to_crs(dong_map.crs)

print("전체 역사-노선 행:", len(metro_gdf))
print("지하철 좌표계:", metro_gdf.crs)

In [ ]:
metro_by_shp = gpd.sjoin(
    metro_gdf,
    dong_boundary,
    how="left",
    predicate="within"
)

metro_seoul = metro_by_shp.loc[
    metro_by_shp["ADSTRD_CD"].notna()
].copy()

print("전체 역사-노선 행:", len(metro_by_shp))
print("서울 SHP 내부 역사-노선 행:", len(metro_seoul))
print("서울 내부 고유 역명:", metro_seoul["역사명"].nunique())
print(
    "지하철이 있는 행정동:",
    metro_seoul["ADSTRD_CD"].nunique()
)

In [ ]:
bus_access = (
    bus_by_shp.loc[
        bus_by_shp["ADSTRD_CD"].notna()
    ]
    .groupby("ADSTRD_CD", as_index=False)
    .agg(
        버스정류소수=("NODE_ID", "nunique")
    )
)

metro_access = (
    metro_seoul
    .groupby("ADSTRD_CD", as_index=False)
    .agg(
        지하철_고유역명수=("역사명", "nunique"),
        지하철_노선역수=("역사_ID", "nunique"),
        지하철_노선수=("호선", "nunique")
    )
)

print("버스정류소가 있는 행정동:", len(bus_access))
print("지하철역이 있는 행정동:", len(metro_access))

In [ ]:
transit_columns = [
    "버스정류소수",
    "지하철_고유역명수",
    "지하철_노선역수",
    "지하철_노선수"
]

index_base = index_base.drop(
    columns=transit_columns,
    errors="ignore"
)

index_base = index_base.merge(
    bus_access,
    on="ADSTRD_CD",
    how="left",
    validate="one_to_one"
)

index_base = index_base.merge(
    metro_access,
    on="ADSTRD_CD",
    how="left",
    validate="one_to_one"
)

index_base[transit_columns] = (
    index_base[transit_columns]
    .fillna(0)
    .astype(int)
)

assert len(index_base) == 425

print("최종 행정동 수:", len(index_base))
print("버스정류소 합계:", index_base["버스정류소수"].sum())
print("역사-노선 합계:", index_base["지하철_노선역수"].sum())

In [ ]:
if index_base.crs is None:
    raise ValueError("index_base의 좌표계가 없습니다.")

if index_base.crs.is_geographic:
    raise ValueError(
        "면적 계산 전에 투영 좌표계가 필요합니다."
    )

index_base["면적_km2"] = (
    index_base.geometry.area / 1_000_000
)

index_base["버스정류소밀도"] = (
    index_base["버스정류소수"]
    / index_base["면적_km2"]
)

index_base["지하철역밀도"] = (
    index_base["지하철_고유역명수"]
    / index_base["면적_km2"]
)

index_base["지하철노선역밀도"] = (
    index_base["지하철_노선역수"]
    / index_base["면적_km2"]
)

display(
    index_base[
        [
            "ADSTRD_NM",
            "면적_km2",
            "버스정류소수",
            "버스정류소밀도",
            "지하철_고유역명수",
            "지하철역밀도",
            "지하철_노선역수"
        ]
    ].sort_values(
        "버스정류소밀도",
        ascending=False
    ).head(20)
)

## 10. Composite Index Methods

### Selected variables and directions

| Pillar | Indicator | Justification | Higher opportunity when |
|---|---|---|---|
| Demand–Supply Opportunity | English institutions per 10,000 residents aged 40–69 | Measures competing supply relative to NADO's primary adult market | Lower |
| Price Opportunity | Median listed tuition | Higher competing tuition may provide more room for price differentiation | Higher |
| Operational Accessibility | Bus-stop density and unique-station density | Dense public transit improves in-person service feasibility | Higher |

The two transit indicators are averaged before aggregation so that the transit pillar receives the same overall weight as supply and price. Skewed nonnegative indicators are transformed with `log1p`, and all inputs are winsorized at the 1st and 99th percentiles.

Three composite-index methods are compared:

1. **Method 1 — Min–Max, equal pillar weights:** each oriented pillar is scaled from 0 to 100 and the three pillars are averaged.
2. **Method 2 — Z-score, equal pillar weights:** each oriented indicator is standardized and the three pillars are averaged.
3. **Method 3 — Mazziotta–Pareto (MPI−):** standardized pillars are aggregated with a penalty for imbalance. Because higher values represent a positive expansion opportunity, the penalty is subtracted.

In [ ]:
required_index_columns = [
    "영어기관_1만명당_20_69",
    "영어기관_1만명당_40_69",
    "수강료_분석값",
    "버스정류소밀도",
    "지하철역밀도"
]

missing_index_columns = [
    col for col in required_index_columns
    if col not in index_base.columns
]
if missing_index_columns:
    raise KeyError(f"Missing index variables: {missing_index_columns}")

assert index_base[required_index_columns].notna().all().all()
index_model = index_base.copy()

# Reduce right-skew while retaining zeros.
index_model["log_supply_40_69"] = np.log1p(
    index_model["영어기관_1만명당_40_69"]
)
index_model["log_supply_20_69"] = np.log1p(
    index_model["영어기관_1만명당_20_69"]
)
index_model["log_bus_density"] = np.log1p(
    index_model["버스정류소밀도"]
)
index_model["log_metro_density"] = np.log1p(
    index_model["지하철역밀도"]
)

def winsorize_series(series, lower=0.01, upper=0.99):
    return series.clip(
        lower=series.quantile(lower),
        upper=series.quantile(upper)
    )

for col in [
    "log_supply_40_69", "log_supply_20_69", "수강료_분석값",
    "log_bus_density", "log_metro_density"
]:
    index_model[f"{col}_w"] = winsorize_series(index_model[col])

display(
    index_model[
        [
            "log_supply_40_69_w", "수강료_분석값_w",
            "log_bus_density_w", "log_metro_density_w"
        ]
    ].describe().T
)

### 10.1 Method 1: Min–Max Scaling with Equal Weights

In [ ]:
def minmax_benefit(series, higher_is_better=True):
    minimum = series.min()
    maximum = series.max()
    if maximum == minimum:
        return pd.Series(50.0, index=series.index)
    score = (series - minimum) / (maximum - minimum) * 100
    return score if higher_is_better else 100 - score

index_model["M1_supply"] = minmax_benefit(
    index_model["log_supply_40_69_w"],
    higher_is_better=False
)
index_model["M1_price"] = minmax_benefit(
    index_model["수강료_분석값_w"],
    higher_is_better=True
)
index_model["M1_bus"] = minmax_benefit(
    index_model["log_bus_density_w"],
    higher_is_better=True
)
index_model["M1_metro"] = minmax_benefit(
    index_model["log_metro_density_w"],
    higher_is_better=True
)
index_model["M1_transit"] = index_model[["M1_bus", "M1_metro"]].mean(axis=1)
index_model["Method1_MinMax"] = index_model[
    ["M1_supply", "M1_price", "M1_transit"]
].mean(axis=1)
index_model["Method1_Rank"] = index_model["Method1_MinMax"].rank(
    ascending=False, method="min"
).astype(int)

### 10.2 Method 2: Z-score Scaling with Equal Weights

In [ ]:
def zscore(series):
    standard_deviation = series.std(ddof=0)
    if standard_deviation == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.mean()) / standard_deviation

index_model["M2_supply"] = -zscore(index_model["log_supply_40_69_w"])
index_model["M2_price"] = zscore(index_model["수강료_분석값_w"])
index_model["M2_bus"] = zscore(index_model["log_bus_density_w"])
index_model["M2_metro"] = zscore(index_model["log_metro_density_w"])
index_model["M2_transit"] = index_model[["M2_bus", "M2_metro"]].mean(axis=1)
index_model["Method2_ZScore"] = index_model[
    ["M2_supply", "M2_price", "M2_transit"]
].mean(axis=1)
index_model["Method2_Rank"] = index_model["Method2_ZScore"].rank(
    ascending=False, method="min"
).astype(int)

### 10.3 Method 3: Mazziotta–Pareto Index (MPI−)

In [ ]:
def mpi_standardize(series):
    return 100 + 10 * zscore(series)

# Standardize the three equally weighted Method 1 pillars around 100.
index_model["M3_supply"] = mpi_standardize(index_model["M1_supply"])
index_model["M3_price"] = mpi_standardize(index_model["M1_price"])
index_model["M3_transit"] = mpi_standardize(index_model["M1_transit"])

mpi_columns = ["M3_supply", "M3_price", "M3_transit"]
index_model["MPI_mean"] = index_model[mpi_columns].mean(axis=1)
index_model["MPI_sd"] = index_model[mpi_columns].std(axis=1, ddof=0)
index_model["MPI_cv"] = index_model["MPI_sd"] / index_model["MPI_mean"]

# MPI-minus penalizes imbalance for a positive phenomenon.
index_model["Method3_MPI"] = (
    index_model["MPI_mean"]
    - index_model["MPI_sd"] * index_model["MPI_cv"]
)
index_model["Method3_Rank"] = index_model["Method3_MPI"].rank(
    ascending=False, method="min"
).astype(int)

## 11. Comparison of the Three Methods

In [ ]:
method_columns = [
    "Method1_MinMax",
    "Method2_ZScore",
    "Method3_MPI"
]

print("Pearson correlation")
display(index_model[method_columns].corr(method="pearson"))

print("Spearman rank correlation")
display(index_model[method_columns].corr(method="spearman"))

comparison_rows = []
for method in method_columns:
    top = index_model.nlargest(5, method)
    bottom = index_model.nsmallest(5, method)
    for position, (_, row) in enumerate(top.iterrows(), start=1):
        comparison_rows.append(
            {
                "Method": method,
                "Group": "Top 5",
                "Position": position,
                "District": row["자치구명"],
                "Dong": row["ADSTRD_NM"],
                "Score": row[method]
            }
        )
    for position, (_, row) in enumerate(bottom.iterrows(), start=1):
        comparison_rows.append(
            {
                "Method": method,
                "Group": "Bottom 5",
                "Position": position,
                "District": row["자치구명"],
                "Dong": row["ADSTRD_NM"],
                "Score": row[method]
            }
        )

method_comparison = pd.DataFrame(comparison_rows)
display(method_comparison)

# Age-range sensitivity check for Method 1.
index_model["M1_supply_20_69"] = minmax_benefit(
    index_model["log_supply_20_69_w"],
    higher_is_better=False
)
index_model["Method1_20_69"] = index_model[
    ["M1_supply_20_69", "M1_price", "M1_transit"]
].mean(axis=1)

age_sensitivity = index_model[
    ["Method1_MinMax", "Method1_20_69"]
].corr(method="spearman").iloc[0, 1]
print("40–69 vs. 20–69 Spearman correlation:", round(age_sensitivity, 4))

## 12. Maps of the Three Composite Indices

Percentile scores are mapped so that the three methods can be compared on the same 0–100 scale. Darker areas represent stronger NADO expansion opportunities.

In [ ]:
for method in method_columns:
    index_model[f"{method}_Percentile"] = (
        index_model[method].rank(pct=True, method="average") * 100
    )

fig, axes = plt.subplots(1, 3, figsize=(21, 8))
map_specs = [
    ("Method1_MinMax_Percentile", "Method 1: Min–Max"),
    ("Method2_ZScore_Percentile", "Method 2: Z-score"),
    ("Method3_MPI_Percentile", "Method 3: MPI−")
]

for axis, (column, title) in zip(axes, map_specs):
    index_model.plot(
        column=column,
        cmap="YlOrRd",
        vmin=0,
        vmax=100,
        linewidth=0.25,
        edgecolor="white",
        legend=True,
        ax=axis
    )
    axis.set_title(title, fontsize=14)
    axis.set_axis_off()

fig.suptitle(
    "NADO Expansion Opportunity by Composite-Index Method",
    fontsize=17,
    y=0.98
)
plt.tight_layout()
plt.show()

## 13. AI-Assisted Addition: LISA Map

Following Walker's PySAL workflow, Local Moran's I is applied to the final MPI score with a row-standardized Queen-contiguity weights matrix and 999 conditional random permutations. PySAL's `p_sim` pseudo-p-value is used with a 0.05 threshold. The map identifies statistically significant spatial clusters and outliers:

- **High–High:** high-opportunity dong surrounded by high-opportunity neighbors;
- **Low–Low:** low-opportunity dong surrounded by low-opportunity neighbors;
- **High–Low / Low–High:** spatial outliers;
- **Not significant:** permutation pseudo-p-value of 0.05 or greater.

The blog uses a static map because the assignment requires a LISA map; Walker's GeoPandas `.explore()` step is an optional interactive presentation layer and does not change the statistic.

If the packages are missing, install them once in a separate notebook cell with `%pip install libpysal esda`.

In [ ]:
# Run once in a separate Jupyter cell if the packages are not installed:
# %pip install libpysal esda

from libpysal import weights
from esda.moran import Moran_Local
from matplotlib.patches import Patch


# Use the same final MPI data and geometry that were used in the index maps.
index_model = (
    index_model
    .dropna(subset=["Method3_MPI"])
    .copy()
    .reset_index(drop=True)
)

# Queen contiguity treats polygons that share an edge or a corner as neighbors.
w = weights.contiguity.Queen.from_dataframe(
    index_model,
    use_index=False,
)
w.transform = "r"  # Each dong's neighbor weights sum to 1.

# Official PySAL Local Moran's I with 999 conditional permutations.
lisa = Moran_Local(
    index_model["Method3_MPI"].astype(float).to_numpy(),
    w,
    transformation="r",
    permutations=999,
    seed=42,
)

index_model["Local_Moran_I"] = lisa.Is
index_model["LISA_p_value"] = lisa.p_sim

# PySAL quadrants: 1=HH, 2=LH, 3=LL, 4=HL.
significant_quadrant = np.where(
    lisa.p_sim < 0.05,
    lisa.q,
    0,
)

cluster_labels = {
    0: "Not significant",
    1: "High–High",
    2: "Low–High",
    3: "Low–Low",
    4: "High–Low",
}

index_model["LISA_cluster"] = (
    pd.Series(significant_quadrant, index=index_model.index)
    .map(cluster_labels)
)

display(index_model["LISA_cluster"].value_counts().to_frame("Dongs"))

# Conventional LISA colors used in the professor's reference workflow.
cluster_colors = {
    "High–High": "#d7191c",
    "Low–Low": "#2c7bb6",
    "High–Low": "#f781bf",
    "Low–High": "#74add1",
    "Not significant": "#d9d9d9",
}

fig, axis = plt.subplots(figsize=(10, 10))
index_model.plot(
    color=index_model["LISA_cluster"].map(cluster_colors),
    linewidth=0.3,
    edgecolor="white",
    ax=axis,
)
axis.set_title(
    "LISA Clusters of the MPI NADO Expansion Opportunity Index",
    fontsize=15,
)
axis.set_axis_off()
axis.legend(
    handles=[
        Patch(facecolor=color, edgecolor="none", label=label)
        for label, color in cluster_colors.items()
    ],
    loc="lower left",
    frameon=True,
)
plt.show()


## 14. Final Results and Limitations

Method 3 (MPI−) is used as the final index because it rewards neighborhoods that perform consistently across demand–supply opportunity, price opportunity, and operational accessibility rather than allowing one very strong pillar to fully offset a weak pillar.

Important limitations are:

- living population is a proxy for potential adult learning demand, not observed enrollment intent;
- adult-specific academy information is sparse, so confirmed English institutions are used as the competing-supply proxy;
- tuition is publicly available for only part of the market, and missing dong values are replaced by district medians;
- transit density measures geographic availability, not actual travel time or instructor schedules;
- keyword-based academy classification may contain false positives or false negatives.

In [ ]:
final_results = index_model[
    [
        "ADSTRD_CD", "자치구명", "ADSTRD_NM",
        "M1_supply", "M1_price", "M1_transit",
        "Method1_MinMax", "Method1_Rank",
        "Method2_ZScore", "Method2_Rank",
        "Method3_MPI", "Method3_Rank",
        "수강료_대체수준", "수강료_자료신뢰도",
        "LISA_cluster", "LISA_p_value"
    ]
].sort_values("Method3_Rank")

print("Final MPI top 5 expansion candidates")
display(final_results.head(5))

print("Final MPI bottom 5")
display(final_results.tail(5).sort_values("Method3_Rank", ascending=False))